# VAE + KMeans Customer Clustering

This notebook reuses the classic recommendation preprocessing, builds the same wide customer feature table, trains the same tabular VAE, and then clusters customers in latent space with KMeans.

The elbow chart is used to choose $k$ later, and the cluster centroids are decoded back through the VAE decoder so we can inspect what each cluster represents.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

ROOT = Path.cwd()
SRC_DIR = ROOT / "classic_methods" / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data_processing.preprocessing import Preprocessor, PreprocessorConfig
from embeddings.customer_features import build_customer_feature_matrix
from embeddings.reductions import preprocess_feature_matrix, train_vae

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

CSV_PATH = ROOT / "LUMEN_DS.csv"
MIN_PURCHASES_PER_CUSTOMER = 50
MIN_ITEMS_PER_CUSTOMER = 5
RANDOM_STATE = 42
LATENT_DIM = 16
VAE_EPOCHS = 80
VAE_BATCH_SIZE = 256
VAE_LR = 1e-3
VAE_BETA = 0.05
K_RANGE = range(2, 13)
CHOSEN_K = 4

In [ ]:
raw_df = pd.read_csv(
    "LUMEN_DS.csv",
    sep="|",
    quotechar='"',
    encoding="utf-16",)

preprocessor = Preprocessor(
    PreprocessorConfig(
        min_num_purchases_per_customer=MIN_PURCHASES_PER_CUSTOMER,
        min_num_items_per_customer=MIN_ITEMS_PER_CUSTOMER,
        map_item_codes=True,
        impute=True,
        date_columns=(),
        drop_price_gt_tx=False,
        drop_missing_customer_id=False,
        drop_missing_item_code=False,
        reindex_items_at_end=False,
    )
)
clean_df, idx2item = preprocessor.fit_transform(raw_df)

feature_df = build_customer_feature_matrix(clean_df).sort_index()

print(f"Raw rows: {len(raw_df):,}")
print(f"Filtered rows: {len(clean_df):,}")
print(f"Customers after filter: {feature_df.shape[0]:,}")
print(f"Feature columns: {feature_df.shape[1]:,}")
display(feature_df.head())

In [ ]:
clean_features, feature_matrix, imputer, scaler = preprocess_feature_matrix(feature_df)

vae_result = train_vae(
    feature_matrix,
    latent_dim=LATENT_DIM,
    beta=VAE_BETA,
    epochs=VAE_EPOCHS,
    batch_size=VAE_BATCH_SIZE,
    learning_rate=VAE_LR,
    random_state=RANDOM_STATE,
)
latent = vae_result.latent_mean

display(vae_result.history.tail())
print("Latent shape:", latent.shape)

In [ ]:
elbow_rows = []
for k in K_RANGE:
    if k >= len(latent):
        continue
    model = KMeans(n_clusters=int(k), n_init=20, random_state=RANDOM_STATE)
    labels = model.fit_predict(latent)
    row = {"k": int(k), "inertia": float(model.inertia_)}
    row["silhouette"] = float(silhouette_score(latent, labels)) if len(np.unique(labels)) > 1 else np.nan
    elbow_rows.append(row)

elbow_df = pd.DataFrame(elbow_rows)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(elbow_df["k"], elbow_df["inertia"], marker="o", linewidth=2)
ax1.axvline(CHOSEN_K, color="black", linestyle="--", alpha=0.6, label=f"chosen k = {CHOSEN_K}")
ax1.set_title("Elbow chart in VAE latent space")
ax1.set_xlabel("Number of clusters (k)")
ax1.set_ylabel("KMeans inertia")
ax1.legend()

display(elbow_df)
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=CHOSEN_K, n_init=20, random_state=RANDOM_STATE)
cluster_labels = kmeans.fit_predict(latent)

device = next(vae_result.model.parameters()).device
centroids_latent = torch.tensor(kmeans.cluster_centers_, dtype=torch.float32, device=device)
vae_result.model.eval()
with torch.no_grad():
    decoded_standard = vae_result.model.decoder(centroids_latent).cpu().numpy()

decoded_features = scaler.inverse_transform(decoded_standard)
centroid_df = pd.DataFrame(
    decoded_features,
    columns=clean_features.columns,
    index=[f"cluster_{i}" for i in range(CHOSEN_K)],
)

cluster_counts = pd.Series(cluster_labels).value_counts().reindex(range(CHOSEN_K), fill_value=0).sort_index()
display(cluster_counts.rename("customer_count").to_frame())
display(centroid_df.round(2))

In [ ]:
decoded_z = pd.DataFrame(
    decoded_standard,
    columns=clean_features.columns,
    index=centroid_df.index,
)

def family_score(z_row: pd.Series, patterns: tuple[str, ...]) -> float:
    matched = z_row[[column for column in z_row.index if any(column.startswith(pattern) for pattern in patterns)]]
    if matched.empty:
        return float("-inf")
    return float(matched.clip(lower=0).sum())

def interpret_cluster(z_row: pd.Series) -> dict[str, object]:
    scores = {
        "high activity / long-tenure buyers": family_score(z_row, ("activity__", "time__")),
        "high spend / margin buyers": family_score(z_row, ("agg__Invoiced price__", "agg__margin_value__", "agg__GM%__")),
        "broad assortment buyers": family_score(z_row, ("activity__unique", "activity__top_item_share")),
        "specialized / concentrated buyers": family_score(z_row, ("share__", "bucketshare__", "bucket_")),
    }
    label = max(scores, key=scores.get)
    top_positive = z_row.sort_values(ascending=False).head(6)
    top_negative = z_row.sort_values(ascending=True).head(6)
    dominant_families = sorted(scores.items(), key=lambda item: item[1], reverse=True)[:2]
    rationale = "Highest decoded positive deviations fall in " + " and ".join(
        f"{family} ({score:.2f})" for family, score in dominant_families
    ) + "."
    return {
        "label": label,
        "rationale": rationale,
        "top_positive": top_positive,
        "top_negative": top_negative,
    }

cluster_profiles = []
for cluster_name in decoded_z.index:
    cluster_id = int(cluster_name.split("_")[-1])
    report = interpret_cluster(decoded_z.loc[cluster_name])
    cluster_profiles.append({
        "cluster": cluster_name,
        "customer_count": int(cluster_counts.loc[cluster_id]),
        "label": report["label"],
        "rationale": report["rationale"],
        "top_positive_features": ", ".join(f"{feature} ({value:.2f})" for feature, value in report["top_positive"].items()),
        "top_negative_features": ", ".join(f"{feature} ({value:.2f})" for feature, value in report["top_negative"].items()),
    })

cluster_profiles_df = pd.DataFrame(cluster_profiles)
display(cluster_profiles_df)

for cluster_name in decoded_z.index:
    cluster_id = int(cluster_name.split("_")[-1])
    report = interpret_cluster(decoded_z.loc[cluster_name])
    print(f"{cluster_name}: {report['label']}")
    print(report["rationale"])
    print("Top positive decoded z-scores:")
    display(report["top_positive"].to_frame("decoded_z").round(2))
    print("Top negative decoded z-scores:")
    display(report["top_negative"].to_frame("decoded_z").round(2))
    print("Customer count:", int(cluster_counts.loc[cluster_id]))
    print("-" * 80)

## How to read the centroid labels

The labels are heuristic. I chose them by looking at the strongest decoded deviations from the population mean and grouping the feature names into families such as activity, time, spend, variety, and product-mix shares.

That is why the interpretation is phrased as a behavior pattern rather than a hard business segment. If you change `CHOSEN_K`, the cluster prototypes and labels update automatically.